# 99 Build the Part 3 planner on the real data

The Part 3 demand-flexibility and battery planner (`web/part3/`) is a single web page that opens straight in a
browser, with no server. It doesn't read the database while it runs: its data are built into the page. This notebook
does that build, on the results of the earlier notebooks:

1. **Export** (`scripts/export_part3_inputs.py`): the hourly loads and one row per customer (segment, programme
   flags, appliance types and confidences, technology scores, list flags, mid-year change flags) from `avangrid.db`
   and the cached arrays, to `onedrive-export/Data/part3/`.
2. **Build** (`scripts/run_part3.py`): the planner's calculations (system peak, flexible loads, battery sizing,
   customer economics, ranking), with the assumptions in `config/part3_assumptions.json`, then the page itself.

At the end, open **`onedrive-export/Data/part3/index.html`** in a browser (the last cell prints a link). The page
lists the top 3,000 customers individually (set `MAX_CUSTOMERS` in the setup cell to change that); its summary
figures cover everyone.

**The page holds real customer data.** It is written inside the git-ignored data folder; don't commit it or share it
outside the event. The committed `web/part3/index.html` uses a synthetic demonstration dataset and is safe to share.

Run after notebooks 10–70 (the export needs `technology_predictions`, `appliance_inventory`, `customer_segments` and
`change_points`). How each step works is described in `web/part3/README.md` and on the page's Methodology tab.

## Setup

In [1]:
import json
import subprocess
import sys
import time
from pathlib import Path

import polars as pl
from IPython.display import Markdown, display

from avangrid import config

ROOT = Path.cwd()
PART3 = config.DATA_DIR / 'part3'
# Customers listed individually on the page (the top-ranked ones). Every summary figure still covers all customers.
# 3,000 gives a page of about 30 MB that opens quickly; 0 lists everyone (about 160 MB, slow to open).
MAX_CUSTOMERS = 3000


def run(script, *args):
    """Run a project script with this notebook's Python, show its output, and stop the notebook if it fails."""
    t = time.time()
    r = subprocess.run([sys.executable, str(ROOT / 'scripts' / script), *map(str, args)], cwd=ROOT,
                       capture_output=True, text=True)
    print(r.stdout[-4000:] + (('\n' + r.stderr[-4000:]) if r.returncode else ''))
    if r.returncode:
        raise RuntimeError(f'{script} failed (exit code {r.returncode})')
    print(f'{script}: done in {time.time() - t:.0f}s')

## 1. Export the inputs from the database

In [2]:
run('export_part3_inputs.py')
manifest = json.loads((PART3 / 'manifest.json').read_text())
{k: v for k, v in manifest.items() if not isinstance(v, (dict, list))}

Loading hourly arrays (cached)...
Filling gaps for 33,310 customers x 365 days...
Joining Part 1 and Part 2 results...
Wrote 33,310 customers x 8,760 hours to /Users/nchaturvedi/Documents/Code/avangrid/onedrive-export/Data/part3
  hourly_kwh.npy 1.17 GB, customers.csv 61 columns

export_part3_inputs.py: done in 7s


{'generated': '2026-11-13T23:18:46',
 'source': 'avangrid-hackathon repo (notebooks 10-70)',
 'n_customers': 33310,
 'n_customers_available': 33310,
 'sample': False,
 'n_hours': 8760,
 'first_date': '2025-09-01',
 'last_date': '2026-08-31'}

## 2. Build the planner

This step runs the whole Part 3 method on every customer and writes three files to `onedrive-export/Data/part3/`:
`app_data.json` (everything the page reads), `index.html` (the page, with that data built in) and `candidates.csv`
(one row per customer with the main results, ranked).

In [3]:
run('run_part3.py', *(['--max-customers', MAX_CUSTOMERS] if MAX_CUSTOMERS else []))

33,310 customers x 8,760 hours
  windows: summer HE [17, 18, 19, 20], winter HE [18, 19, 20, 21]; binding season summer  (1s)
  1,000/33,310 customers  (2s)
  2,000/33,310 customers  (3s)
  3,000/33,310 customers  (4s)
  4,000/33,310 customers  (6s)
  5,000/33,310 customers  (7s)
  6,000/33,310 customers  (8s)
  7,000/33,310 customers  (9s)
  8,000/33,310 customers  (11s)
  9,000/33,310 customers  (12s)
  10,000/33,310 customers  (13s)
  11,000/33,310 customers  (14s)
  12,000/33,310 customers  (15s)
  13,000/33,310 customers  (17s)
  14,000/33,310 customers  (18s)
  15,000/33,310 customers  (19s)
  16,000/33,310 customers  (20s)
  17,000/33,310 customers  (22s)
  18,000/33,310 customers  (23s)
  19,000/33,310 customers  (24s)
  20,000/33,310 customers  (25s)
  21,000/33,310 customers  (27s)
  22,000/33,310 customers  (28s)
  23,000/33,310 customers  (29s)
  24,000/33,310 customers  (30s)
  25,000/33,310 customers  (31s)
  26,000/33,310 customers  (33s)
  27,000/33,310 customers  (34s)

## 3. Check the results and open the page

The top of the customer ranking, and the page's location. Open the page in any browser; it needs no internet
connection or server.

In [4]:
page = PART3 / 'index.html'
candidates = pl.read_csv(PART3 / 'candidates.csv')
print(f'{candidates.height:,} customers ranked; page {page.stat().st_size / 1e6:.1f} MB')
with pl.Config(tbl_cols=12, tbl_width_chars=200, tbl_hide_dataframe_shape=True):
    print(candidates.head(10))
display(Markdown(f'**Open the planner:** [{page}]({page.resolve().as_uri()})'))

33,310 customers ranked; page 28.8 MB
┌──────┬──────────┬─────────────────────────────────┬────────────────┬───────┬────────┬───┬─────────────┬───────────────┬──────┬────────┬─────────┬─────────────────────────────────┐
│ rank ┆ customer ┆ segment                         ┆ action         ┆ units ┆ wave   ┆ … ┆ p_heat_pump ┆ p_resist_heat ┆ p_ac ┆ p_pool ┆ p_solar ┆ reason                          │
│ ---  ┆ ---      ┆ ---                             ┆ ---            ┆ ---   ┆ ---    ┆   ┆ ---         ┆ ---           ┆ ---  ┆ ---    ┆ ---     ┆ ---                             │
│ i64  ┆ i64      ┆ str                             ┆ str            ┆ i64   ┆ str    ┆   ┆ f64         ┆ f64           ┆ f64  ┆ f64    ┆ f64     ┆ str                             │
╞══════╪══════════╪═════════════════════════════════╪════════════════╪═══════╪════════╪═══╪═════════════╪═══════════════╪══════╪════════╪═════════╪═════════════════════════════════╡
│ 1    ┆ 30474    ┆ Large, steady homes             

**Open the planner:** [/Users/nchaturvedi/Documents/Code/avangrid/onedrive-export/Data/part3/index.html](file:///Users/nchaturvedi/Documents/Code/avangrid/onedrive-export/Data/part3/index.html)

## Tried but not used

| Tried | Why it isn't used |
|---|---|
| **A live web app reading the database** (the earlier battery simulator: a FastAPI server, `app/server.py`, and its page, `web/index.html`, plus a Streamlit prototype) | Removed in favour of the Part 3 planner. A page with its data built in needs no server, port or running database, so judges only need a browser, and it can be opened on a machine without the data. The trade-off: changing an assumption in `config/part3_assumptions.json` means re-running this notebook. |